In [1]:
import polars as pl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel, 
    AutoTokenizer, 
    Trainer, 
    TrainingArguments,
    BertForSequenceClassification,
    BertTokenizer
)
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from huggingface_hub import snapshot_download
import warnings
import math


warnings.filterwarnings('ignore')

In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

ACCEPTABLE_ACTIONS = ["view", "click", "clickout", "like"]
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000
TOP_K = 15
BATCH_SIZE_TRAIN = 5000      
MAX_SEQ_LEN = 30             
MIN_SEQ_LEN = 3
TOP_N_ITEMS = 5000          
SAMPLE_USERS = 5000         
SAMPLE_RATIO = 0.1 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/marketplace/**"]
)

Fetching ... files: 0it [00:00, ?it/s]

'/kaggle/working'

In [4]:
events_path = "/kaggle/working/dataset/small/marketplace/events"
items_path = "/kaggle/working/dataset/small/marketplace/items.pq"

events_lf = pl.scan_parquet(events_path).select([
    "timestamp", "user_id", "item_id", "action_type", "subdomain", "os"
])

In [5]:
items_lf = pl.scan_parquet(items_path).select([
    "item_id", "brand_id", "category", "subcategory", "price"
])

In [6]:
items_lf = items_lf.drop_nulls(subset=["price"]).with_columns([
    pl.col("category").fill_null("unknown"),
    pl.col("subcategory").fill_null("unknown"),
    pl.col("brand_id").fill_null(-1),
])


In [7]:
top_items = (
    events_lf
    .group_by("item_id")
    .agg(pl.len().alias("cnt"))
    .sort("cnt", descending=True)
    .limit(TOP_N_ITEMS)
    .select("item_id")
    .collect()
)["item_id"].to_list()

In [8]:
events_filtered = (
    events_lf
    .filter(pl.col("item_id").is_in(top_items))
    .filter(pl.col("action_type").is_in(["view", "click", "clickout", "like"]))
    .collect(streaming=True)  
)

In [9]:
action_counts = events_filtered.group_by("action_type").agg(pl.len())
total_events = action_counts["len"].sum()

action_weights = {}
for row in action_counts.iter_rows(named=True):
    weight = math.log(total_events / row["len"]) + 1
    action_weights[row["action_type"]] = weight

In [10]:
events_filtered = events_filtered.with_columns([
    pl.col("action_type").replace(action_weights).alias("target")
])

In [11]:
all_users = events_filtered["user_id"].unique().to_list()
n_users = len(all_users)

if len(all_users) > SAMPLE_USERS:
    np.random.seed(SEED)
    sampled_users = np.random.choice(all_users, SAMPLE_USERS, replace=False)
    events_filtered = events_filtered.filter(pl.col("user_id").is_in(sampled_users))

In [12]:
def global_temporal_split(df_lf: pl.LazyFrame, test_size: float = 0.2):
    """
    Временное разделение данных на train и test.
    """
    # Находим границы времени
    min_ts, max_ts = (
        df_lf
        .select(
            pl.col("timestamp").min().alias("min_ts"),
            pl.col("timestamp").max().alias("max_ts")
        )
        .collect()
        .row(0)
    )
    
    # Вычисляем пороговое значение для разделения
    ts_range = max_ts - min_ts
    split_ts = min_ts + ts_range * (1 - test_size)
    
    # Разделяем данные
    train_lf = df_lf.filter(pl.col("timestamp") < split_ts)
    test_lf = df_lf.filter(pl.col("timestamp") >= split_ts)
    
    return train_lf, test_lf

In [13]:
def ndcg_at_k(user_based_df, relevancy_col, true_items_col,
              predicted_items_col, predicted_score_col, top_k=15):
    """
    Расчет NDCG@k.
    """
    def compute_ndcg(true_items, pred_items, relevancy):
        relevance_dict = dict(zip(true_items, relevancy))
        pred_relevance = [relevance_dict.get(item, 0) for item in pred_items[:top_k]]
        
        dcg = 0
        for i, rel in enumerate(pred_relevance):
            dcg += rel / math.log2(i + 2)
        
        sorted_relevance = sorted(relevancy, reverse=True)[:top_k]
        idcg = 0
        for i, rel in enumerate(sorted_relevance):
            idcg += rel / math.log2(i + 2)
        
        return dcg / idcg if idcg > 0 else 0
    
    ndcg_scores = []
    if isinstance(user_based_df, pl.DataFrame):
        rows = user_based_df.iter_rows(named=True)
    else:
        rows = user_based_df.to_dict('records')
    
    for row in rows:
        ndcg = compute_ndcg(row[true_items_col], row[predicted_items_col], row[relevancy_col])
        ndcg_scores.append(ndcg)
    
    return np.mean(ndcg_scores)

In [14]:
def calculate_metrics(df, k=15):
    """
    Расчет Precision@k и Recall@k.
    """
    precisions = []
    recalls = []
    
    if isinstance(df, pl.DataFrame):
        rows = df.iter_rows(named=True)
    else:
        rows = df.to_dict('records')
    
    for row in rows:
        true_set = set(row["true_items"])
        pred_set = set(row["predicted_items"][:k])
        
        true_positives = len(true_set.intersection(pred_set))
        precision = true_positives / k if k > 0 else 0
        precisions.append(precision)
        
        recall = true_positives / len(true_set) if len(true_set) > 0 else 0
        recalls.append(recall)
    
    return np.mean(precisions), np.mean(recalls)

In [15]:
def build_sequences_streaming(events_df, max_seq_len=30, min_seq_len=3):
    """
    Построение последовательностей с использованием потоковой обработки.
    """
    # Сортировка
    events_df = events_df.sort(["user_id", "timestamp"])
    
    sequences = []
    current_user = None
    current_items = []
    current_targets = []
    
    for row in tqdm(events_df.iter_rows(named=True), total=len(events_df), desc="Building sequences"):
        user_id = row["user_id"]
        
        if user_id != current_user:
            # Сохраняем последовательности для предыдущего пользователя
            if len(current_items) >= min_seq_len:
                for i in range(min_seq_len, len(current_items) + 1):
                    sequences.append({
                        "user_id": current_user,
                        "item_sequence": current_items[:i],
                        "target": current_targets[i-1],
                        "seq_len": i
                    })
            
            # Начинаем нового пользователя
            current_user = user_id
            current_items = [row["item_id"]]
            current_targets = [row["target"]]
        else:
            current_items.append(row["item_id"])
            current_targets.append(row["target"])
            
            # Ограничиваем длину
            if len(current_items) > max_seq_len:
                current_items = current_items[-max_seq_len:]
                current_targets = current_targets[-max_seq_len:]
    
    # Последний пользователь
    if len(current_items) >= min_seq_len:
        for i in range(min_seq_len, len(current_items) + 1):
            sequences.append({
                "user_id": current_user,
                "item_sequence": current_items[:i],
                "target": current_targets[i-1],
                "seq_len": i
            })
    
    print(f"Построено {len(sequences)} последовательностей")
    return sequences

In [16]:
timestamp_threshold = events_filtered["timestamp"].quantile(0.8)
train_events = events_filtered.filter(pl.col("timestamp") < timestamp_threshold)
test_events = events_filtered.filter(pl.col("timestamp") >= timestamp_threshold)

In [17]:
train_sequences = build_sequences_streaming(train_events, MAX_SEQ_LEN, MIN_SEQ_LEN)
test_sequences = build_sequences_streaming(test_events, MAX_SEQ_LEN, MIN_SEQ_LEN)


Building sequences: 100%|██████████| 240626/240626 [00:01<00:00, 167670.79it/s]


Построено 77490 последовательностей



Building sequences: 100%|██████████| 60157/60157 [00:00<00:00, 355910.57it/s]

Построено 29753 последовательностей


In [18]:
item_to_idx = {item: idx for idx, item in enumerate(top_items)}
num_items = len(top_items)

In [19]:
class EfficientRecDataset(Dataset):
    
    def __init__(self, sequences, item_to_idx, max_seq_len=30):
        self.max_seq_len = max_seq_len
        self.item_to_idx = item_to_idx
        
        # Преобразуем в эффективный формат
        self.data = []
        for seq in sequences:
            item_indices = [item_to_idx.get(item, 0) for item in seq["item_sequence"]]
            seq_len = len(item_indices)
            
            # Padding
            padded_seq = item_indices + [0] * (max_seq_len - seq_len)
            attention_mask = [1] * seq_len + [0] * (max_seq_len - seq_len)
            
            self.data.append({
                "input_ids": padded_seq,
                "attention_mask": attention_mask,
                "target": seq["target"],
                "seq_len": seq_len
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "target": torch.tensor(item["target"], dtype=torch.float)
        }

In [20]:
train_dataset = EfficientRecDataset(train_sequences, item_to_idx, MAX_SEQ_LEN)
test_dataset = EfficientRecDataset(test_sequences, item_to_idx, MAX_SEQ_LEN)

In [21]:
class PositionalEncoding(nn.Module):
    """Синусоидальное позиционное кодирование из оригинального Transformer"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

In [22]:
class TransformerRanker(nn.Module):
    """Полноценная Transformer модель для ранжирования"""
    
    def __init__(self, num_items, d_model=128, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.3, max_seq_len=50):
        super().__init__()
        
        self.d_model = d_model
        self.item_embedding = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len)
        
        # Настоящий Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.dropout = nn.Dropout(dropout)
        
        # Prediction head
        self.predictor = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, input_ids, attention_mask):
        batch_size, seq_len = input_ids.shape
        
        # Embeddings + Positional Encoding
        x = self.item_embedding(input_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.dropout(x)
        
        # Causal mask (запрещаем видеть будущее)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1).to(x.device)
        
        # Transformer forward
        src_key_padding_mask = (attention_mask == 0)
        transformer_out = self.transformer(x, mask=causal_mask, src_key_padding_mask=src_key_padding_mask)
        
        # Используем выход последнего токена
        seq_lengths = attention_mask.sum(dim=1) - 1
        batch_indices = torch.arange(batch_size, device=input_ids.device)
        last_output = transformer_out[batch_indices, seq_lengths]
        
        return self.predictor(last_output).squeeze(-1)

In [23]:
# Параметры
BATCH_SIZE = 32
MAX_SEQ_LEN = 30


# Энкодинг item_id -> индексы
item_to_idx = {item: idx + 1 for idx, item in enumerate(top_items)}  # 0 для padding
num_items = len(top_items) + 1

def create_dataloaders(train_sequences, test_sequences, item_to_idx, max_seq_len, batch_size):
    """Создание DataLoader для обучения"""
    
    class TransformerDataset(Dataset):
        def __init__(self, sequences, item_to_idx, max_seq_len):
            self.max_seq_len = max_seq_len
            self.item_to_idx = item_to_idx
            self.data = []
            
            for seq in sequences:
                target_value = seq["target"]
                if isinstance(target_value, str):
                    target_value = float(target_value) if target_value.replace('.', '').isdigit() else 0.0
                elif not isinstance(target_value, (int, float)):
                    target_value = 0.0
                
                item_indices = [item_to_idx.get(item, 0) for item in seq["item_sequence"]]
                seq_len = len(item_indices)
                
                padded_items = item_indices + [0] * (max_seq_len - seq_len)
                attention_mask = [1] * seq_len + [0] * (max_seq_len - seq_len)
                
                self.data.append({
                    "input_ids": torch.tensor(padded_items, dtype=torch.long),
                    "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                    "target": torch.tensor(target_value, dtype=torch.float),
                    "user_id": seq["user_id"]
                })
        
        def __len__(self):
            return len(self.data)
        
        def __getitem__(self, idx):
            return self.data[idx]
    
    train_dataset = TransformerDataset(train_sequences, item_to_idx, max_seq_len)
    test_dataset = TransformerDataset(test_sequences, item_to_idx, max_seq_len)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    return train_loader, test_loader, len(item_to_idx) + 1


try:
    train_loader, test_loader, num_items = create_dataloaders(
        train_sequences, test_sequences, item_to_idx, MAX_SEQ_LEN, BATCH_SIZE
    )
    print(f"Train loader: {len(train_loader)} батчей")
    print(f"Test loader: {len(test_loader)} батчей")
    print(f"Num items: {num_items}")
except NameError:
    print("ОШИБКА: train_sequences и test_sequences не определены!")
    print("Сначала запустите блок построения последовательностей.")

Train loader: 2422 батчей
Test loader: 930 батчей
Num items: 5001


In [24]:

model = TransformerRanker(
    num_items=num_items,
    d_model=64,
    nhead=4,
    num_layers=3,
    dim_feedforward=256,
    dropout=0.3,
    max_seq_len=MAX_SEQ_LEN
).to(device)


criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=0.001,           # learning rate
    weight_decay=1e-5   # L2 regularization
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5,         
    patience=3
)

In [ ]:

EPOCHS = 30
best_val_loss = float('inf')
best_val_ndcg = 0
train_losses = []
val_losses = []
val_ndcgs = []

# Для группировки предсказаний по пользователям (для метрик)
def aggregate_predictions_by_user(predictions_df):
    """Агрегирует предсказания по пользователям для расчета NDCG"""
    return (
        predictions_df
        .group_by("user_id")
        .agg([
            pl.col("item_id").alias("predicted_items"),
            pl.col("pred_score").alias("predicted_scores")
        ])
    )

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, targets)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    

    model.eval()
    val_loss = 0
    
    # Собираем предсказания для расчета метрик
    val_predictions = []  # для агрегации по пользователям
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["target"].to(device)
            
            outputs = model(input_ids, attention_mask)
            val_loss += criterion(outputs, targets).item()
            
            
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    # Обновляем learning rate
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.6f}")
    print(f"  Val Loss:   {avg_val_loss:.6f}")
    print(f"  LR: {current_lr:.6f}")
    
    # Сохраняем лучшую модель
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_transformer_model.pt")
        print(f"  -> Новый лучший чекпоинт! Val Loss: {best_val_loss:.6f}")
    


Epoch 1/30 [Train]:  79%|███████▊  | 1905/2422 [00:27<00:06, 75.74it/s, loss=0.0089]

Epoch 4/30 [Train]: 100%|██████████| 2422/2422 [00:34<00:00, 71.03it/s, loss=0.0086]

Epoch 4/30 [Val]: 100%|██████████| 930/930 [00:02<00:00, 399.74it/s]



Epoch 4/30
  Train Loss: 0.398202
  Val Loss:   0.459530
  LR: 0.001000



Epoch 5/30 [Train]: 100%|██████████| 2422/2422 [00:32<00:00, 74.45it/s, loss=0.6780]

Epoch 5/30 [Val]: 100%|██████████| 930/930 [00:02<00:00, 405.14it/s]



Epoch 5/30
  Train Loss: 0.396732
  Val Loss:   0.458949
  LR: 0.001000
  -> Новый лучший чекпоинт! Val Loss: 0.458949



Epoch 25/30 [Train]: 100%|██████████| 2422/2422 [00:40<00:00, 59.27it/s, loss=0.6808]

Epoch 25/30 [Val]: 100%|██████████| 930/930 [00:02<00:00, 406.19it/s]



Epoch 25/30
  Train Loss: 0.397246
  Val Loss:   0.459172
  LR: 0.000031



Epoch 26/30 [Train]:  40%|███▉      | 968/2422 [00:13<00:19, 74.89it/s, loss=0.7646]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)


Epoch 28/30 [Train]:  40%|███▉      | 957/2422 [00:12<00:19, 75.13it/s, loss=0.7645]


In [36]:
action_weight_map = {"view": 1.0, "click": 2.0, "like": 3.0, "clickout": 4.0}

test_user_items = (
    test_events
    .with_columns(pl.col("action_type").replace(action_weight_map).alias("weight"))
    .group_by("user_id")
    .agg([
        pl.col("item_id").alias("true_items"),
        pl.col("weight").alias("relevancy"),
    ])
)

model.eval()
user_item_predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        user_ids = batch["user_id"]
        outputs = model(input_ids, attention_mask).cpu().numpy()
        
        for idx, user_id in enumerate(user_ids):
            seq_len = attention_mask[idx].sum().item()
            last_item_id = input_ids[idx, seq_len - 1].item()
            idx_to_item = {v: k for k, v in item_to_idx.items()}
            item_id = idx_to_item.get(last_item_id, last_item_id)
            
            user_item_predictions.append({
                "user_id": user_id.item(),
                "item_id": item_id,
                "pred_score": float(outputs[idx])
            })

TOP_K = 15
predictions_df = pl.DataFrame(user_item_predictions)

user_predictions_topk = (
    predictions_df
    .group_by("user_id")
    .agg([
        pl.col("item_id").sort_by("pred_score", descending=True).head(TOP_K).alias("predicted_items"),
        pl.col("pred_score").sort_by("pred_score", descending=True).head(TOP_K).alias("predicted_scores")
    ])
)

user_metrics_df = user_predictions_topk.join(test_user_items, on="user_id", how="inner")

def compute_precision_at_k(true_items, pred_items, k=15):
    if not pred_items:
        return 0.0
    return len(set(true_items) & set(pred_items[:k])) / k

def compute_recall_at_k(true_items, pred_items, k=15):
    if not true_items:
        return 0.0
    return len(set(true_items) & set(pred_items[:k])) / len(true_items)

def compute_ndcg_at_k(true_items, relevancy, pred_items, k=15):
    relevance_dict = dict(zip(true_items, relevancy))
    pred_relevance = [float(relevance_dict.get(item, 0)) for item in pred_items[:k]]
    
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(pred_relevance) if rel > 0)
    sorted_relevance = sorted([float(r) for r in relevancy], reverse=True)[:k]
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(sorted_relevance) if rel > 0)
    
    return dcg / idcg if idcg > 0 else 0.0

def compute_rmse(preds, targets):
    return np.sqrt(np.mean((np.array(preds) - np.array(targets)) ** 2))

def compute_mae(preds, targets):
    return np.mean(np.abs(np.array(preds) - np.array(targets)))

precisions, recalls, ndcgs = [], [], []
pred_scores, true_scores = [], []

for row in user_metrics_df.iter_rows(named=True):
    if not row["predicted_items"] or not row["true_items"]:
        continue
    
    precisions.append(compute_precision_at_k(row["true_items"], row["predicted_items"], TOP_K))
    recalls.append(compute_recall_at_k(row["true_items"], row["predicted_items"], TOP_K))
    ndcgs.append(compute_ndcg_at_k(row["true_items"], row["relevancy"], row["predicted_items"], TOP_K))
    
    for item, pred_score in zip(row["predicted_items"], row["predicted_scores"]):
        if item in row["true_items"]:
            idx = row["true_items"].index(item)
            pred_scores.append(pred_score)
            true_scores.append(float(row["relevancy"][idx]))

results = pl.DataFrame({
    "target": ["target"],
    "latent_factors": [num_items - 1],  
    "top_k": [TOP_K],
    "ndcg": [np.mean(ndcgs) if ndcgs else 0.0],
    "precision": [np.mean(precisions) if precisions else 0.0],
    "recall": [np.mean(recalls) if recalls else 0.0],
    "rmse": [compute_rmse(pred_scores, true_scores) if pred_scores else 0.0],
    "mae": [compute_mae(pred_scores, true_scores) if pred_scores else 0.0],
})

print(results)


ФИНАЛЬНАЯ ОЦЕНКА МОДЕЛИ



Inference: 100%|█████████▉| 927/930 [00:13<00:00, 70.57it/s]
                                                            


РЕЗУЛЬТАТЫ ОЦЕНКИ
shape: (1, 8)
┌────────┬────────────────┬───────┬──────────┬───────────┬─────────┬──────────┬──────────┐
│ target ┆ latent_factors ┆ top_k ┆ ndcg     ┆ precision ┆ recall  ┆ rmse     ┆ mae      │
│ ---    ┆ ---            ┆ ---   ┆ ---      ┆ ---       ┆ ---     ┆ ---      ┆ ---      │
│ str    ┆ i64            ┆ i64   ┆ f64      ┆ f64       ┆ f64     ┆ f64      ┆ f64      │
╞════════╪════════════════╪═══════╪══════════╪═══════════╪═════════╪══════════╪══════════╡
│ target ┆ 5000           ┆ 15    ┆ 0.821932 ┆ 0.52858   ┆ 0.50071 ┆ 0.276701 ┆ 0.178435 │
└────────┴────────────────┴───────┴──────────┴───────────┴─────────┴──────────┴──────────┘


In [ ]:
print(1)